# Ingest: NYC Yellow Taxi 2024

## Overview

This notebook loads all 12 months of NYC Yellow Taxi trip data (January through December 2024) from parquet files into PostgreSQL. The raw data is ingested into the trips_raw table without any cleaning or transformation. This preserves the original data for reproducibility and allows cleaning to be done in the next phase.

We also load the TLC taxi zone lookup table, which maps LocationID codes to borough and zone names. This enables later analysis by geographic area.

## What This Notebook Does

Input:
- 12 parquet files: yellow_tripdata_2024_01.parquet through yellow_tripdata_2024_12.parquet (Taxi Trip data for the whole of 2024)
- 1 CSV file: taxi_zone_lookup.csv (zone metadata)

Process:
1. Connect to PostgreSQL using SQLAlchemy
2. Read each parquet file into a pandas DataFrame
3. Insert data into PostgreSQL in chunks (100,000 rows at a time) for memory efficiency
4. Load the zone lookup table
5. Verify row counts and date ranges to see if everything is okay

Output:
- trips_raw table: approximately 36,300,000 rows (all 12 months combined)
- taxi_zones table: 265 rows (location reference data)

## Key Considerations

- The data is raw and has not been cleaned yet, strictly just loading in this notebook
- Large files are inserted in 100,000 row batches so there aren't any memory issues
- Tables are indexed on frequently queried columns like pickup time and Location ID's to improve query performance
- The zone lookup table is seperate and normalized to a different table to efficiently store the zone names

## 0. Imports & Configuration

In [ ]:
#   IMPORTS
import pandas as pd
import os
from sqlalchemy import create_engine, text
from pathlib import Path
import time



#   CONFIG
DB_PASSWORD = 'user'
DB_USER     = 'postgres'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'NYC Yellow Taxi Trip Report Data 2024'



#   PATHS
DATA_RAW_DIR       = Path('C:/users/matth/nyc-taxi-demand-pricing-analysis/data/raw')
ZONE_LOOKUP_PATH   = Path('C:/users/matth/nyc-taxi-demand-pricing-analysis/data/taxi_zone_lookup.csv')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

#   CoNNECTIONS TEST
with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('ConnecteL:', result.scalar())

✅ Connected to PostgreSQL: PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


## 1. Load Zone Lookup → `taxi_zones`

265 rows mapping LocationID → Borough, Zone, service_zone.

In [2]:
zones_df = pd.read_csv(ZONE_LOOKUP_PATH)

print(f'Shape:   {zones_df.shape}')
print(f'Columns: {zones_df.columns.tolist()}')
print()
zones_df.head()

Shape:   (265, 4)
Columns: ['LocationID', 'Borough', 'Zone', 'service_zone']



,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [ ]:
# LOAD ZONES TO POSTGRES
zones_df.to_sql('taxi_zones', con=engine, if_exists='replace', index=False)

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM taxi_zones')).scalar()

print(f'loaded: {count} rows')

✅ taxi_zones loaded: 265 rows


## 2. Load 12 Months of Trip Data → `trips_raw`

Each month is around 2–4M rows. We insert in chunks of 100k rows to keep memory usage low.

In [ ]:
# DATA FILES PREVIEW
parquet_files = sorted(DATA_RAW_DIR.glob('yellow_tripdata_2024-*.parquet'))
print(f'Found {len(parquet_files)} parquet files:')
for f in parquet_files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f'  {f.name}  ({size_mb:.0f} MB)')

if len(parquet_files) != 12:
    print('error')

Found 12 parquet files:
  yellow_tripdata_2024-01.parquet  (48 MB)
  yellow_tripdata_2024-02.parquet  (48 MB)
  yellow_tripdata_2024-03.parquet  (57 MB)
  yellow_tripdata_2024-04.parquet  (56 MB)
  yellow_tripdata_2024-05.parquet  (60 MB)
  yellow_tripdata_2024-06.parquet  (57 MB)
  yellow_tripdata_2024-07.parquet  (50 MB)
  yellow_tripdata_2024-08.parquet  (49 MB)
  yellow_tripdata_2024-09.parquet  (58 MB)
  yellow_tripdata_2024-10.parquet  (61 MB)
  yellow_tripdata_2024-11.parquet  (58 MB)
  yellow_tripdata_2024-12.parquet  (59 MB)


In [5]:
# INGESTION INTO POSTGRES
CHUNK_SIZE  = 100_000   DB rows insert batch amt
total_rows  = 0
start_time  = time.time()

for i, parquet_file in enumerate(parquet_files, start=1):
    t0 = time.time()
    df = pd.read_parquet(parquet_file)
    file_rows = len(df)
    mode = 'replace' if i == 1 else 'append'

    df.to_sql(
        'trips_raw',
        con=engine,
        if_exists=mode,
        index=False,
        chunksize=CHUNK_SIZE,
        method='multi'
    )

    total_rows += file_rows
    elapsed = time.time() - t0
    print(f'[{i:02d}/12] {parquet_file.name}  |  {file_rows:>8,} rows  |  {elapsed:.0f}s  |  running total: {total_rows:,}')

total_elapsed = time.time() - start_time
print(f'\n Done {total_rows:,} ')

[01/12] yellow_tripdata_2024-01.parquet  |  2,964,624 rows  |  795s  |  running total: 2,964,624
[02/12] yellow_tripdata_2024-02.parquet  |  3,007,526 rows  |  791s  |  running total: 5,972,150
[03/12] yellow_tripdata_2024-03.parquet  |  3,582,628 rows  |  24448s  |  running total: 9,554,778
[04/12] yellow_tripdata_2024-04.parquet  |  3,514,289 rows  |  917s  |  running total: 13,069,067
[05/12] yellow_tripdata_2024-05.parquet  |  3,723,833 rows  |  1061s  |  running total: 16,792,900
[06/12] yellow_tripdata_2024-06.parquet  |  3,539,193 rows  |  1539s  |  running total: 20,332,093
[07/12] yellow_tripdata_2024-07.parquet  |  3,076,903 rows  |  1526s  |  running total: 23,408,996
[08/12] yellow_tripdata_2024-08.parquet  |  2,979,183 rows  |  1377s  |  running total: 26,388,179
[09/12] yellow_tripdata_2024-09.parquet  |  3,633,030 rows  |  2253s  |  running total: 30,021,209
[10/12] yellow_tripdata_2024-10.parquet  |  3,833,771 rows  |  1408s  |  running total: 33,854,980
[11/12] yellow_

## 3. Verify in PostgreSQL

In [6]:
with engine.connect() as conn:
    trips_count = conn.execute(text('SELECT COUNT(*) FROM trips_raw')).scalar()
    zones_count = conn.execute(text('SELECT COUNT(*) FROM taxi_zones')).scalar()

print(f'trips_raw  → {trips_count:,} rows  (expected ~36M)')
print(f'taxi_zones → {zones_count:,} rows  (expected 265)')

# VERIFY IF ROW AMOUNT IS NORMAL
if trips_count < 30_000_000:
    print('low trip_raw rows')
else:
    print('Success')

trips_raw  → 41,169,663 rows  (expected ~36M)
taxi_zones → 265 rows  (expected 265)
✅ Row count looks good!


In [7]:
# CHECKING DATE RANGE
date_check = pd.read_sql_query('''
    SELECT
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        COUNT(DISTINCT DATE_TRUNC('month', tpep_pickup_datetime)) AS months_present
    FROM trips_raw
''', con=engine)

print('Date range:')
date_check

Date range check:


,earliest_pickup,latest_pickup,months_present
0,2002-12-31 22:17:43,2026-06-26 23:53:12,19


In [8]:
# DATA PREVIEW
sample = pd.read_sql_query('SELECT * FROM trips_raw LIMIT 5', con=engine)
print('Sample Data:')
sample

Sample rows from trips_raw:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0
